# Notebook 03: ESCO Occupation-Skill Baseline

This notebook generates the ESCO Occupation-Skill Baseline as a skill foundation for each ESCO occupation. The result is saved in data/interim and used in Notebook 04 to derive the KldB future profiles:
- loads tables from 01: esco_occupations.parquet; esco_skills.parquet; occupation_skill_relations.parquet
- joins tables into a long base table, each row: (ESCO occupation, skill, relation_type)
- performs quality and structural checks
- saves result: data/interim/occupation_skills_baseline.parquet
- The baseline remains in long format (one row = ESCO occupation × skill)

(Sources as in the previous notebooks)

## 1. Imports & Project Paths

In [1]:
from pathlib import Path
import pandas as pd

# Projektwurzel
PROJECT_ROOT = Path.cwd().resolve().parent
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Interim path:", DATA_INTERIM)
print("Processed path:", DATA_PROCESSED)

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

Project root: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung
Interim path: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\interim
Processed path: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed


Load base tables from Notebook 01, Parquet files

In [ ]:
occ_path   = DATA_INTERIM / "esco_occupations.parquet"
skills_path = DATA_INTERIM / "esco_skills.parquet"
rels_path   = DATA_INTERIM / "occupation_skill_relations.parquet"

# Load
esco_occupations = pd.read_parquet(occ_path)
esco_skills = pd.read_parquet(skills_path)
occupation_skill_relations = pd.read_parquet(rels_path)

print("esco_occupations:", esco_occupations.shape)
print("esco_skills:", esco_skills.shape)
print("occupation_skill_relations:", occupation_skill_relations.shape)

display(esco_occupations.head(3))
display(esco_skills.head(3))
display(occupation_skill_relations.head(3))

esco_occupations: (3039, 6)
esco_skills: (13939, 6)
occupation_skill_relations: (129004, 3)


,occupation_uri,occupation_code,isco_group,pref_label_en,alt_labels_en,description_en
0,http://data.europa.eu/esco/occupation/00030d09...,2654.1.7,2654,technical director,[technical and operations director\nhead of te...,Technical directors realise the artistic visio...
1,http://data.europa.eu/esco/occupation/000e93a3...,8121.4,8121,metal drawing machine operator,[metal drawing machine technician\nmetal drawi...,Metal drawing machine operators set up and ope...
2,http://data.europa.eu/esco/occupation/0019b951...,7543.10.3,7543,precision device inspector,[inspector of precision instruments\nprecision...,Precision device inspectors make sure precisio...


,skill_uri,pref_label_en,alt_labels_en,description_en,reuse_level,skill_type
0,http://data.europa.eu/esco/skill/0005c151-5b5a...,manage musical staff,[manage staff of music\ncoordinate duties of m...,Assign and manage staff tasks in areas such as...,sector-specific,skill/competence
1,http://data.europa.eu/esco/skill/00064735-8fad...,supervise correctional procedures,[oversee prison procedures\nmanage correctiona...,Supervise the operations of a correctional fac...,occupation-specific,skill/competence
2,http://data.europa.eu/esco/skill/000709ed-2be5...,apply anti-oppressive practices,[apply non-oppressive practices\napply an anti...,"Identify oppression in societies, economies, c...",sector-specific,skill/competence


,occupation_uri,skill_uri,relation_type
0,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/fed5b267-73fa...,essential
1,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/05bc7677-5a64...,essential
2,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/271a36a0-bc7a...,essential


The number of relations (occupation_skill_relations) determines the maximum size of the baseline, since each relation will later generate exactly one row in long format.

Consistency check: all URIs in relations must also be present in the master tables

In [3]:
occ_in_rel = set(occupation_skill_relations["occupation_uri"])
occ_all = set(esco_occupations["occupation_uri"])
missing_occ = occ_in_rel - occ_all

skill_in_rel = set(occupation_skill_relations["skill_uri"])
skill_all = set(esco_skills["skill_uri"])
missing_skills = skill_in_rel - skill_all

print("URIs in relations, die nicht in occupations vorkommen:", len(missing_occ))
print("URIs in relations, die nicht in skills vorkommen:", len(missing_skills))

URIs in relations, die nicht in occupations vorkommen: 0
URIs in relations, die nicht in skills vorkommen: 0


0 missing URIs, meaning that relations can be fully integrated referentially, with no mapping breaks between the relation table and the master tables.

## 2. Create an Occupation-Skill Baseline

Join relations with occupations: Add occupation information to the relations table. `many_to_one` ensures that each relation is assigned to exactly one occupation 
and prevents unintended duplicates.

In [ ]:
# Relations + Occupations
occ_rel = occupation_skill_relations.merge(
    esco_occupations,
    on="occupation_uri",
    how="left",
    validate="many_to_one" # Every relation has exactly one occupation
)

print("Relations + Occupations:", occ_rel.shape)
occ_rel.head(5)

Relations + Occupations: (129004, 8)


,occupation_uri,skill_uri,relation_type,occupation_code,isco_group,pref_label_en,alt_labels_en,description_en
0,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/fed5b267-73fa...,essential,2654.1.7,2654,technical director,[technical and operations director\nhead of te...,Technical directors realise the artistic visio...
1,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/05bc7677-5a64...,essential,2654.1.7,2654,technical director,[technical and operations director\nhead of te...,Technical directors realise the artistic visio...
2,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/271a36a0-bc7a...,essential,2654.1.7,2654,technical director,[technical and operations director\nhead of te...,Technical directors realise the artistic visio...
3,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/47ed1d37-971b...,essential,2654.1.7,2654,technical director,[technical and operations director\nhead of te...,Technical directors realise the artistic visio...
4,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/591dd514-735b...,essential,2654.1.7,2654,technical director,[technical and operations director\nhead of te...,Technical directors realise the artistic visio...


In [ ]:
# Change the names of the job categories
occ_rel = occ_rel.rename(columns={
    "pref_label_en": "occupation_title_en",
    "alt_labels_en": "occupation_alt_labels_en",
    "description_en": "occupation_description_en",
})

Build skills

In [ ]:
occ_skill_full = occ_rel.merge(
    esco_skills,
    on="skill_uri",
    how="left",
    suffixes=("", "_skill"), # in case of name conflicts
    validate="many_to_one" # Each relation has exactly one skill
)

print("Relations + Occupations + Skills:", occ_skill_full.shape)
occ_skill_full.head(5)

Relations + Occupations + Skills: (129004, 13)


,occupation_uri,skill_uri,relation_type,occupation_code,isco_group,occupation_title_en,occupation_alt_labels_en,occupation_description_en,pref_label_en,alt_labels_en,description_en,reuse_level,skill_type
0,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/fed5b267-73fa...,essential,2654.1.7,2654,technical director,[technical and operations director\nhead of te...,Technical directors realise the artistic visio...,theatre techniques,[theatre technique\ntheatre approaches\ntheatr...,The techniques that facilitate a successful pr...,sector-specific,knowledge
1,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/05bc7677-5a64...,essential,2654.1.7,2654,technical director,[technical and operations director\nhead of te...,Technical directors realise the artistic visio...,organise rehearsals,[organise rehearsal\norganize rehearsals\nplan...,"Manage, schedule and run rehearsals for the pe...",sector-specific,skill/competence
2,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/271a36a0-bc7a...,essential,2654.1.7,2654,technical director,[technical and operations director\nhead of te...,Technical directors realise the artistic visio...,write risk assessment on performing arts produ...,[write assessment of risks about performing ar...,"Assess risks, propose improvements and describ...",sector-specific,skill/competence
3,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/47ed1d37-971b...,essential,2654.1.7,2654,technical director,[technical and operations director\nhead of te...,Technical directors realise the artistic visio...,coordinate with creative departments,[liaise with other artistic departments\ncoord...,Coordinate activities with other artistic and ...,sector-specific,skill/competence
4,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/591dd514-735b...,essential,2654.1.7,2654,technical director,[technical and operations director\nhead of te...,Technical directors realise the artistic visio...,adapt to artists' creative demands,[meet demands by creative artists\nadapt to de...,"Work with artists, striving to understand the ...",sector-specific,skill/competence


Change column names

In [7]:
occ_skill_full = occ_skill_full.rename(columns={
    "pref_label_en": "skill_title_en",
    "alt_labels_en": "skill_alt_labels_en",
    "description_en": "skill_description_en",
})

## 3. Clean up relevant columns

Target schema: The resulting schema contains both job and skill metadata as well as the relationship type, thereby forming the complete structured baseline.

In [8]:
occupation_skills_baseline = occ_skill_full[[
    # Beruf
    "occupation_uri","occupation_code","isco_group","occupation_title_en","occupation_alt_labels_en","occupation_description_en",
    # Skill
    "skill_uri","skill_title_en","skill_alt_labels_en","skill_description_en","reuse_level","skill_type",
    # Relation
    "relation_type",
]]

Remove whitespace and NaN

In [ ]:
text_cols = ["occupation_title_en","occupation_alt_labels_en","occupation_description_en","skill_title_en","skill_alt_labels_en","skill_description_en","relation_type","reuse_level","skill_type",]

for col in text_cols:
    if col in occupation_skills_baseline.columns:
        occupation_skills_baseline[col] = (
            occupation_skills_baseline[col]
            .fillna("").astype(str).str.strip() # Text fields are normalized as strings (NaN → "", Trim)
        )

print("occupation_skills_baseline:", occupation_skills_baseline.shape)
occupation_skills_baseline.head(5)

occupation_skills_baseline: (129004, 13)


,occupation_uri,occupation_code,isco_group,occupation_title_en,occupation_alt_labels_en,occupation_description_en,skill_uri,skill_title_en,skill_alt_labels_en,skill_description_en,reuse_level,skill_type,relation_type
0,http://data.europa.eu/esco/occupation/00030d09...,2654.1.7,2654,technical director,['technical and operations director\nhead of t...,Technical directors realise the artistic visio...,http://data.europa.eu/esco/skill/fed5b267-73fa...,theatre techniques,['theatre technique\ntheatre approaches\ntheat...,The techniques that facilitate a successful pr...,sector-specific,knowledge,essential
1,http://data.europa.eu/esco/occupation/00030d09...,2654.1.7,2654,technical director,['technical and operations director\nhead of t...,Technical directors realise the artistic visio...,http://data.europa.eu/esco/skill/05bc7677-5a64...,organise rehearsals,['organise rehearsal\norganize rehearsals\npla...,"Manage, schedule and run rehearsals for the pe...",sector-specific,skill/competence,essential
2,http://data.europa.eu/esco/occupation/00030d09...,2654.1.7,2654,technical director,['technical and operations director\nhead of t...,Technical directors realise the artistic visio...,http://data.europa.eu/esco/skill/271a36a0-bc7a...,write risk assessment on performing arts produ...,['write assessment of risks about performing a...,"Assess risks, propose improvements and describ...",sector-specific,skill/competence,essential
3,http://data.europa.eu/esco/occupation/00030d09...,2654.1.7,2654,technical director,['technical and operations director\nhead of t...,Technical directors realise the artistic visio...,http://data.europa.eu/esco/skill/47ed1d37-971b...,coordinate with creative departments,['liaise with other artistic departments\ncoor...,Coordinate activities with other artistic and ...,sector-specific,skill/competence,essential
4,http://data.europa.eu/esco/occupation/00030d09...,2654.1.7,2654,technical director,['technical and operations director\nhead of t...,Technical directors realise the artistic visio...,http://data.europa.eu/esco/skill/591dd514-735b...,adapt to artists' creative demands,"[""meet demands by creative artists\nadapt to d...","Work with artists, striving to understand the ...",sector-specific,skill/competence,essential


## 4. Baseline Quality Checks

In [ ]:
# Distribution of relation_type
print("relation_type counts:")
print(occupation_skills_baseline["relation_type"].value_counts(dropna=False))

relation_type counts:
relation_type
essential    67622
optional     61382
Name: count, dtype: int64


The nearly balanced distribution shows that ESCO systematically models both essential and optional competencies.

In [ ]:
# Number of skills per occupation
skills_per_occ = (
    occupation_skills_baseline
    .groupby("occupation_uri")["skill_uri"]
    .nunique()
)

print("Anzahl ESCO-Berufe:", skills_per_occ.shape[0])
print("Skills pro Beruf (nunique(skill_uri)):")
print(skills_per_occ.describe())

Anzahl ESCO-Berufe: 3039
Skills pro Beruf (nunique(skill_uri)):
count    3039.000000
mean       42.443896
std        25.754169
min         7.000000
25%        28.000000
50%        37.000000
75%        49.000000
max       345.000000
Name: skill_uri, dtype: float64


- The variation in the number of skills per occupation reflects differences in ESCO granularity
- Outliers should/can be taken into account in subsequent aggregations/interpretations.

In [ ]:
# Examples of occupational skill profiles
example_occ = occupation_skills_baseline["occupation_uri"].iloc[0]

print("Beispiel-Beruf:", example_occ)
display(
    occupation_skills_baseline
    .loc[occupation_skills_baseline["occupation_uri"] == example_occ,
         ["occupation_code", "occupation_title_en", "skill_title_en", "relation_type", "reuse_level", "skill_type"]]
    .head(20)
)

Beispiel-Beruf: http://data.europa.eu/esco/occupation/00030d09-2b3a-4efd-87cc-c4ea39d27c34


,occupation_code,occupation_title_en,skill_title_en,relation_type,reuse_level,skill_type
0,2654.1.7,technical director,theatre techniques,essential,sector-specific,knowledge
1,2654.1.7,technical director,organise rehearsals,essential,sector-specific,skill/competence
2,2654.1.7,technical director,write risk assessment on performing arts produ...,essential,sector-specific,skill/competence
3,2654.1.7,technical director,coordinate with creative departments,essential,sector-specific,skill/competence
4,2654.1.7,technical director,adapt to artists' creative demands,essential,sector-specific,skill/competence
5,2654.1.7,technical director,negotiate health and safety issues with third ...,essential,cross-sector,skill/competence
6,2654.1.7,technical director,adapt designers’ work to the performance venue,essential,sector-specific,skill/competence
7,2654.1.7,technical director,promote health and safety,essential,sector-specific,skill/competence
8,2654.1.7,technical director,coordinate technical teams in artistic product...,essential,sector-specific,skill/competence
9,2654.1.7,technical director,write technical riders,optional,sector-specific,skill/competence


relation_type can be used later for weighting (e.g., "essential" is given greater weight than "optional")

## 5. List of Skills by Occupation, Aggregated Version

The aggregated table is provided solely for a concise overview. The analytical basis remains the occupation_skills_baseline table in its long format.

In [13]:
occupation_skills_agg = (
    occupation_skills_baseline
    .groupby(["occupation_uri","occupation_code","occupation_title_en",])
    .apply(lambda df: df[["skill_uri", "skill_title_en", "relation_type"]].to_dict("records"))
    .reset_index(name="skills")
)

print("Aggregierte Tabelle:", occupation_skills_agg.shape)
occupation_skills_agg.head(3)

Aggregierte Tabelle: (3039, 4)


C:\Users\sigle\AppData\Local\Temp\ipykernel_30180\2603120331.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: df[["skill_uri", "skill_title_en", "relation_type"]].to_dict("records"))


,occupation_uri,occupation_code,occupation_title_en,skills
0,http://data.europa.eu/esco/occupation/00030d09...,2654.1.7,technical director,[{'skill_uri': 'http://data.europa.eu/esco/ski...
1,http://data.europa.eu/esco/occupation/000e93a3...,8121.4,metal drawing machine operator,[{'skill_uri': 'http://data.europa.eu/esco/ski...
2,http://data.europa.eu/esco/occupation/0019b951...,7543.10.3,precision device inspector,[{'skill_uri': 'http://data.europa.eu/esco/ski...


## 6. Saving the Baseline

Save the Long table as the default and as an interim version; technical basis

In [14]:
baseline_path = DATA_INTERIM / "occupation_skills_baseline.parquet"
occupation_skills_baseline.to_parquet(baseline_path, index=False)

print("Gespeichert in:", baseline_path)

Gespeichert in: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\interim\occupation_skills_baseline.parquet


In [ ]:
# aggregated list
agg_path = DATA_INTERIM / "occupation_skills_agg.parquet"
occupation_skills_agg.to_parquet(agg_path, index=False)

print("Aggregierte Version gespeichert in:", agg_path)

Aggregierte Version gespeichert in: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\interim\occupation_skills_agg.parquet


# Conclusion Notebook 03

- An occupation-skill baseline was created from the ESCO base tables (esco_occupations, esco_skills, occupation_skill_relations)
- The occupation_skills_baseline table is in long format: each row links an ESCO occupation to a skill, including the relationship type (essential/optional) and metadata
- Data quality verified via consistency checks and statistical analyses
- Baseline saved in data/interim/occupation_skills_baseline.parquet and forms the basis for deriving KldB future profiles in NB 04